# Bài tập Điểm cộng: Xây dựng Complex Data Generator

Đây là bộ mô phỏng dữ liệu (Data Simulator) hoàn chỉnh để chuẩn bị cho:
- **Tuần 5:** Chạy thử A/A Test và kiểm tra Covariate Balance.
- **Tuần 6:** Thực hành Observational Studies (Matching, Adjustment).

Chúng ta sẽ mô phỏng các biến (Covariates) có ý nghĩa kinh tế thật sự dựa trên EDA Tuần 1.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Cài đặt seed để đảm bảo Reproducibility
np.random.seed(42)
sns.set_theme(style="whitegrid")

## 1. Xây dựng DataGenerator Class

Class này đẻ ra 1 tập dữ liệu User-level với các đặc điểm phức tạp đan chéo nhau.

In [ ]:
class ComplexDataGenerator:
    def __init__(self, n_users=20000, true_ate=1.5):
        self.n_users = n_users
        self.true_ate = true_ate
        self.df = None
        
    def generate_covariates(self):
        """Sinh ra các biến nhiễu/đặc trưng của User"""
        # 1. Tuổi tài khoản (Tenure) tính bằng tháng: Từ 1 đến 60 tháng, skew phải
        tenure = np.random.lognormal(mean=2.5, sigma=0.8, size=self.n_users).astype(int)
        tenure = np.clip(tenure, 1, 60)
        
        # 2. Vị trí sống (Khu vực trung tâm Manhattan hay ngoại ô)
        # Khách cũ (tenure cao) có xác suất ở trung tâm cao hơn
        prob_urban = np.clip(0.3 + (tenure / 60) * 0.4, 0.1, 0.9)
        is_urban = np.random.binomial(1, prob_urban)
        
        # 3. Mức độ hoạt động: Số ngày không mở app (Recency)
        # Khách Urban thường mở app nhiều hơn -> Recency thấp hơn
        recency_mean = np.where(is_urban == 1, 3, 10)
        recency = np.random.poisson(recency_mean)
        recency = np.clip(recency, 0, 30)
        
        self.df = pd.DataFrame({
            'user_id': range(1, self.n_users + 1),
            'tenure_months': tenure,
            'is_urban': is_urban,
            'recency_days': recency
        })
        
    def generate_outcomes(self):
        """Sinh ra Base Rides, Treatment Assignment và Observed Outcome"""
        # Base rides Y0: Phụ thuộc cực mạnh vào is_urban và recency
        base_rides = 2 + (self.df['is_urban'] * 5) + (self.df['tenure_months'] * 0.1) - (self.df['recency_days'] * 0.2)
        base_rides = np.clip(np.random.normal(base_rides, 1.5), 0, None).round().astype(int)
        self.df['base_rides'] = base_rides
        
        # Treatment assignment (Confounding): Ưu tiên phát voucher cho dân Urban và Khách lâu năm
        prob_t = 0.1 + (self.df['is_urban'] * 0.4) + (self.df['tenure_months'] / 100)
        prob_t = np.clip(prob_t, 0.05, 0.95)
        self.df['treatment_obs'] = np.random.binomial(1, p=prob_t)
        
        # Random Assignment (A/B Test 50/50)
        self.df['treatment_rand'] = np.random.binomial(1, p=0.5, size=self.n_users)
        
        # Outcome
        noise = np.random.normal(0, 1, self.n_users)
        self.df['Y_obs'] = np.clip(self.df['base_rides'] + (self.true_ate * self.df['treatment_obs']) + noise, 0, None).round().astype(int)
        self.df['Y_rand'] = np.clip(self.df['base_rides'] + (self.true_ate * self.df['treatment_rand']) + noise, 0, None).round().astype(int)
        
    def run(self):
        self.generate_covariates()
        self.generate_outcomes()
        return self.df

In [ ]:
# Khởi tạo bộ dữ liệu 20,000 khách hàng
generator = ComplexDataGenerator(n_users=20000, true_ate=1.5)
df = generator.run()
print(df.shape)
display(df.head())

## 2. Kiểm chứng Covariate Balance (Lợi ích cho Tuần 5)

Trong Tuần 5, chúng ta sẽ phải chạy A/A Test để kiểm tra xem hệ thống Randomization có tốt không.
Một Randomization tốt phải đảm bảo **Covariate Balance**: Các biến nhiễu (Tuổi tài khoản, Khu vực) phải dàn đều giữa nhóm Treatment và Control.

In [ ]:
def check_covariate_balance(data, treatment_col):
    covariates = ['tenure_months', 'is_urban', 'recency_days', 'base_rides']
    balance_df = data.groupby(treatment_col)[covariates].mean().T
    balance_df.columns = ['Control (T=0)', 'Treatment (T=1)']
    balance_df['Diff (%)'] = ((balance_df['Treatment (T=1)'] - balance_df['Control (T=0)']) / balance_df['Control (T=0)']) * 100
    return balance_df.round(2)

print("1. KIỂM TRA BALANCE TRÊN DỮ LIỆU BỊ NHIỄU (OBSERVATIONAL):")
display(check_covariate_balance(df, 'treatment_obs'))

print("\n2. KIỂM TRA BALANCE TRÊN DỮ LIỆU A/B TEST (RANDOMIZED):")
display(check_covariate_balance(df, 'treatment_rand'))

**Nhận xét cực kỳ quan trọng:**
- Ở dữ liệu Bị nhiễu (Observational): Dân Urban chiếm tỷ trọng quá lớn ở nhóm Treatment (gấp 3.8 lần). Tuổi tài khoản cũng cao hơn. Việc lấy Y_obs trừ nhau chắc chắn sẽ ra Bias cực lớn.
- Ở dữ liệu A/B Test (Randomized): Độ chênh lệch (Diff %) cực kỳ nhỏ (chỉ 0-1%). Dữ liệu dàn đều hoàn hảo. Đây chính là tiêu chuẩn Vàng của A/A Test trong Tuần 5!

In [ ]:
# Export data để dùng cho Tuần 5, 6
import os
os.makedirs('../../data/processed', exist_ok=True)
df.to_csv('../../data/processed/complex_simulation_data.csv', index=False)
print("Exported dataset successfully!")